# A10 · Training at Scale on GPUs & AWS

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adib2024/aws-architect-path/blob/main/notebooks/a10_scale.ipynb)

Large models don't fit on one GPU, and training them takes months of GPU time. Learn the memory equation, mixed precision, gradient accumulation, the parallelism strategies, compute-optimal scaling laws, and how to launch a distributed job on SageMaker.

Part of the **AI Engineer Path** (https://adib2024.github.io/aws-architect-path/#a10). In Colab, use **Runtime → Change runtime type → T4 GPU** for anything that trains.

**You will:**
- Compute the training memory budget: weights, gradients, optimizer state and activations.
- Use bf16 autocast, gradient accumulation and gradient clipping.
- Choose between data, tensor, pipeline and sharded (ZeRO/FSDP) parallelism.
- Plan a run with Chinchilla scaling and estimate time and cost on AWS.

## The math

### Memory for mixed-precision AdamW

$$\text{bytes/param} = \underbrace{2}_{\text{bf16 weights}} + \underbrace{2}_{\text{bf16 grads}} + \underbrace{4}_{\text{fp32 master}} + \underbrace{4 + 4}_{\text{Adam } m,\,v} = 16$$

Before any activations. This is why 7B models don't fit on a single 80 GB GPU for full fine-tuning.

**Worked example**

1. 7B params $\times$ 16 B $= 112$ GB (plus activations)
2. FSDP / ZeRO-3 over 8 GPUs: $112/8 = 14$ GB per GPU (fits on 8×A100 40 GB with room for activations)

### Activation memory per layer

$$M_{act} \approx s\,b\,h\left(34 + \frac{5\,a\,s}{h}\right) \text{ bytes (16-bit)}$$

From Korthikanti et al. (2022): $s$ sequence length, $b$ batch, $h$ hidden size, $a$ heads. The $5as/h$ term is the attention matrix, which FlashAttention removes.

**Worked example**

1. GPT-2 small, $s = 1024,\ b = 8,\ h = 768,\ a = 12$: $34 + 80 = 114$
2. $1024\cdot8\cdot768\cdot114 = 717$ MB per layer $\times 12 = 8.6$ GB
3. with FlashAttention (34 only): 2.6 GB (activation checkpointing cuts it further)

### Compute-optimal training (Chinchilla)

$$D^* \approx 20\,N, \qquad C = 6ND, \qquad t = \frac{C}{n_{gpu}\cdot P_{peak}\cdot \text{MFU}}$$

MFU (model FLOPs utilisation) is the fraction of peak you actually achieve: 35–55% is good at scale.

**Worked example**

1. $N = 7\times10^9 \Rightarrow D^* = 1.4\times10^{11}$ tokens
2. $C = 6\cdot7\times10^9\cdot1.4\times10^{11} = 5.9\times10^{21}$ FLOPs
3. 64 A100s at 312 TFLOP/s and 40% MFU: $5.9\times10^{21}/8.0\times10^{15} \approx 7.4\times10^{5}$ s (≈ 8.5 days)
4. modern practice trains far beyond $20N$ (e.g. 1–15T tokens for 7–8B models) (smaller models are cheaper to serve, so extra training pays off)

In [ ]:
%pip install -q sagemaker

## Lab: Mixed precision, accumulation and a SageMaker job

Measure memory savings locally, then package a DDP training script for SageMaker.

### Step 1

The memory and compute planner as code.

In [ ]:
def train_memory_gb(n_params, bytes_per_param=16, shards=1):
    return n_params * bytes_per_param / shards / 1e9

def chinchilla_days(n_params, gpus, peak_tflops=312, mfu=0.4):
    tokens = 20 * n_params
    flops = 6 * n_params * tokens
    return tokens, flops, flops / (gpus * peak_tflops * 1e12 * mfu) / 86400

for n in [124e6, 1.3e9, 7e9, 70e9]:
    t, c, days = chinchilla_days(n, gpus=64)
    print(f"{n/1e9:6.2f}B  mem {train_memory_gb(n):7.1f} GB  tokens {t:.1e}  FLOPs {c:.1e}  {days:7.1f} days on 64xA100")

### Step 2

Mixed-precision autocast (bf16 where supported, otherwise fp16 with a loss scaler), gradient accumulation, clipping, and measured peak memory.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
device = 'cuda'
model = nn.Sequential(*[nn.Sequential(nn.Linear(2048, 8192), nn.GELU(), nn.Linear(8192, 2048)) for _ in range(6)]).to(device)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M params")
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1)

amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16   # T4 -> fp16
def run(precision, accum=4, micro=16):
    use_amp = precision == 'amp'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp and amp_dtype == torch.float16)  # fp16 needs loss scaling
    torch.cuda.reset_peak_memory_stats()
    for step in range(5):
        for _ in range(accum):
            x = torch.randn(micro, 512, 2048, device=device)
            with torch.autocast(device_type='cuda', dtype=amp_dtype, enabled=use_amp):
                loss = model(x).float().pow(2).mean()
            scaler.scale(loss / accum).backward()   # scale so the sum equals the mean over the big batch
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
    return torch.cuda.max_memory_allocated() / 1e9

print("fp32 peak GB:", round(run('fp32'), 2))
print(f"{amp_dtype} autocast peak GB:", round(run('amp'), 2))

### Step 3

A DDP training script. Save as `train_ddp.py`; run locally with `torchrun --nproc_per_node=<gpus> train_ddp.py`.

In [ ]:
%%writefile train_ddp.py
import os, torch, torch.nn as nn, torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

def main():
    dist.init_process_group("nccl")
    rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(rank)
    model = nn.Sequential(nn.Linear(1024, 4096), nn.GELU(), nn.Linear(4096, 1024)).cuda()
    model = DDP(model, device_ids=[rank])
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
    steps = int(os.environ.get("SM_HP_STEPS", 200))
    for step in range(steps):
        x = torch.randn(32, 1024, device="cuda")
        loss = model(x).pow(2).mean()
        opt.zero_grad(); loss.backward(); opt.step()     # DDP all-reduces grads inside backward()
        if step % 50 == 0 and dist.get_rank() == 0:
            print(f"step {step} loss {loss.item():.4f} world_size {dist.get_world_size()}")
    if dist.get_rank() == 0:
        os.makedirs(os.environ.get("SM_MODEL_DIR", "model"), exist_ok=True)
        torch.save(model.module.state_dict(), os.path.join(os.environ.get("SM_MODEL_DIR", "model"), "model.pt"))
    dist.destroy_process_group()

if __name__ == "__main__":
    main()

### Step 4

Launch it on SageMaker with 4 GPUs (ml.g5.12xlarge = 4× A10G). Needs `pip install sagemaker` and a SageMaker execution role. Check your GPU quota for this instance type first.

> ⚠️ This step runs against your AWS account and may incur charges. Skip it if you are only using Colab.

In [ ]:
import sagemaker
from sagemaker.pytorch import PyTorch
role = "arn:aws:iam::<ACCOUNT_ID>:role/<SageMakerExecutionRole>"
est = PyTorch(
    entry_point="train_ddp.py", source_dir=".", role=role,
    framework_version="2.3", py_version="py311",
    instance_type="ml.g5.12xlarge", instance_count=1,
    distribution={"torch_distributed": {"enabled": True}},     # launches torchrun for you
    hyperparameters={"steps": 500},
    max_run=3600,                                             # hard stop after 1 hour
)
est.fit(wait=True)
print(est.model_data)     # s3://.../model.tar.gz

### Clean up

SageMaker training jobs stop and release GPUs when they finish. Check **SageMaker → Training jobs** shows Completed or Stopped, and delete large checkpoints from S3 that you don't need.

## Check yourself

**1. Full fine-tuning a 13B model with mixed-precision AdamW needs roughly how much memory before activations?**

A. 26 GB  
B. 52 GB  
C. 208 GB  
D. 13 GB

<details><summary>Answer</summary>

**C.** $13\times10^9\cdot16$ B $= 208$ GB, hence sharding (FSDP/ZeRO) or parameter-efficient methods (A11).

</details>

**2. Chinchilla-optimal tokens for a 3B model?**

A. 3B  
B. 60B  
C. 600B  
D. 20B

<details><summary>Answer</summary>

**B.** $D \approx 20N = 60\times10^9$.

</details>

**3. What does DDP communicate after each backward pass?**

A. The input data  
B. Gradients (averaged with all-reduce)  
C. The optimizer state  
D. Nothing

<details><summary>Answer</summary>

**B.** Each rank computes gradients on its own data; all-reduce averages them so every replica applies the same update.

</details>